In [ ]:
import os
import requests
import pandas as pd
from openai import OpenAI
from chromadb import Client
from chromadb.config import Settings
import gradio as gr

secrets_path = "../.secrets"  
WEATHER_API_KEY = None
API_GATEWAY_KEY = None

with open(secrets_path, "r") as f:
    for line in f:
        if line.strip() == "" or line.startswith("#"):
            continue
        key, value = line.strip().split("=", 1)
        if key == "WEATHER_API_KEY":
            WEATHER_API_KEY = value
        elif key == "API_GATEWAY_KEY":
            API_GATEWAY_KEY = value


client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    api_key='any value',  # dummy
    default_headers={"x-api-key": API_GATEWAY_KEY}
)

def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding


def weather_service(city: str) -> str:
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {"q": city, "appid": WEATHER_API_KEY, "units": "metric"}
    r = requests.get(url, params=params)
    data = r.json()
    if r.status_code != 200 or "main" not in data:
        return f"Could not find weather for '{city}'. Please check the city name."
    temp = data["main"]["temp"]
    desc = data["weather"][0]["description"]
    humidity = data["main"]["humidity"]
    return f"Currently in {city}: {desc}, {temp:.1f}°C, humidity {humidity}%."


# Load SQuAD CSV
df = pd.read_csv("../train-squad.csv")  
docs = []
for _, row in df.iterrows():
    context = row["context"]
    question = row["question"]
    answer = row.get("answer", "")
    docs.append(f"{context} Question: {question} Answer: {answer}")

# Initialize Chroma v0.3.29
persist_dir = "../chroma_db"
os.makedirs(persist_dir, exist_ok=True)

chroma_client = Client(
    Settings(
        chroma_db_impl="duckdb+parquet",
        persist_directory=persist_dir
    )
)

collection = chroma_client.get_or_create_collection("squad_docs")

# Add embeddings 
if len(collection.get()) == 0:
    for i, doc in enumerate(docs):
        emb = get_embedding(doc)
        collection.add(
            ids=[str(i)],
            documents=[doc],
            embeddings=[emb]
        )
    chroma_client.persist()

def semantic_search_service(query: str) -> str:
    emb = get_embedding(query)
    results = collection.query(query_embeddings=[emb], n_results=3)
    docs_found = results['documents'][0] if results['documents'] else []
    return "\n\n".join(docs_found) if docs_found else "No relevant results found."


def joke_service(_):
    return "Why did the AI cross the road? To optimize the other side!"


chat_memory = []

def chat_system(user_input):
    chat_memory.append(f"User: {user_input}")

    # Guardrails
    forbidden = ["cats", "dogs", "horoscope", "zodiac", "taylor swift"]
    if any(word in user_input.lower() for word in forbidden):
        response = "Sorry, I cannot answer questions about that topic."
    elif "weather" in user_input.lower():
        # extract city
        city = user_input.split("in")[-1].strip() if "in" in user_input.lower() else "London"
        response = weather_service(city)
    elif "joke" in user_input.lower():
        response = joke_service(user_input)
    else:
        response = semantic_search_service(user_input)

    chat_memory.append(f"Bot: {response}")
    return "\n".join(chat_memory[-6:])  # show last 3 exchanges

iface = gr.Interface(
    fn=chat_system,
    inputs="text",
    outputs="text",
    title="Assignment Chatbot",
    description="A chat system with weather, semantic search, and a joke service. Friendly personality!"
)

iface.launch()